# Alfred-Coder — Fine-tune on YOUR work (Colab, free GPU)

Fine-tunes **Qwen3-Coder-30B-A3B-Instruct** (Alfred-Coder's base) on your own (prompt -> good-output) pairs using Unsloth QLoRA, then exports a **GGUF** you load in LM Studio.

**Before running:** Runtime -> Change runtime type -> **GPU** (T4 = free tier).

**Honest caveats:**
- The 30B MoE on a free T4 (16 GB) is *tight*. If you hit OOM: (a) use **Colab Pro** (L4/A100) for the training step (~a few $/mo), or (b) set `MODEL_NAME` to a smaller Qwen3-Coder variant AND run that same model in LM Studio (the fine-tune must match the model you run).
- Fine-tuning personalizes to *your style*; it does not make the model generally smarter.
- Training runs HERE on Colab, never on your CPU.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen3-Coder-30B-A3B-Instruct"  # Alfred-Coder base
# If a free T4 OOMs: use Colab Pro, OR switch to a smaller Qwen3-Coder here
# AND run that same smaller model in LM Studio (fine-tune must match run model).
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
    dtype = None,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

In [ ]:
from google.colab import files
from datasets import load_dataset

print("Upload train.jsonl (built by scripts/build-finetune-jsonl.ps1):")
up = files.upload()            # choose train.jsonl
fname = list(up.keys())[0]

ds = load_dataset("json", data_files=fname, split="train")

def fmt(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}

ds = ds.map(fmt)
print("examples:", len(ds))
print(ds[0]["text"][:500])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        seed = 42,
        output_dir = "outputs",
    ),
)
trainer.train()

In [ ]:
# Merge LoRA into the base and export a GGUF (Q4_K_M) for LM Studio. Can be slow for 30B.
model.save_pretrained_gguf("alfred-coder", tokenizer, quantization_method = "q4_k_m")

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("alfred-coder-gguf", "zip", "alfred-coder")
files.download("alfred-coder-gguf.zip")

## Load into LM Studio
1. Unzip `alfred-coder-gguf.zip`.
2. LM Studio -> My Models -> reveal the models folder; copy the `.gguf` into `alfred-coder/`.
3. Reload LM Studio, select **alfred-coder**, start the local server (`http://localhost:1234/v1`).
4. Alfred's local-coder will call it there (see `docs/local-coder/LM-STUDIO-SETUP.md`).